# 00 — Data Pipeline
**Global Supply Chain Intelligence**

This notebook walks through the complete data ingestion pipeline:
1. FRED API macro indicators (or synthetic fallback)
2. UN Comtrade trade flow data (or synthetic fallback)
3. Synthetic manufacturing data (SKUs, demand, disruption events)
4. DuckDB schema creation and data loading
5. Feature engineering SQL views

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import duckdb
from src.ingest import run_pipeline
from src.synthetic import save_all

## Run Full Pipeline

In [ ]:
# Run the full data pipeline
all_good = run_pipeline()

## Inspect DuckDB Tables

In [ ]:
con = duckdb.connect('../data/processed/supply_chain.db', read_only=True)

# Table counts
tables = ['macro_indicators', 'trade_flows', 'skus', 'weekly_demand', 'disruption_events']
for t in tables:
    count = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t}: {count:,} rows')

In [ ]:
# Sample macro indicators
con.execute('SELECT * FROM macro_indicators LIMIT 10').fetchdf()

In [ ]:
# SKU distribution by category
con.execute("""
    SELECT category, COUNT(*) as count, 
           AVG(lead_time_days) as avg_lead_time,
           AVG(unit_cost_usd) as avg_cost
    FROM skus GROUP BY category ORDER BY count DESC
""").fetchdf()

In [ ]:
# Disruption events
con.execute('SELECT * FROM disruption_events').fetchdf()

In [ ]:
# Feature engineering views
print('\n--- Supplier Concentration Index (HHI) ---')
display(con.execute('SELECT * FROM supplier_concentration_index').fetchdf())

print('\n--- Top Disrupted Commodities ---')
display(con.execute("""
    SELECT * FROM commodity_disruption_score 
    WHERE ABS(z_score) > 1.5 ORDER BY ABS(z_score) DESC LIMIT 10
""").fetchdf())

In [ ]:
con.close()